# RegQuest ML: 99 - Machine Learning Experiments
### Hyperparameter Tuning, Tree Depth Analysis, and Feature Importance

---
## 1. Overview
Tests hyperparameter variations for Random Forest and feature contributions.


In [ ]:
import sys
from pathlib import Path
import numpy as np
from sklearn.ensemble import RandomForestRegressor
ML_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(ML_ROOT) not in sys.path:
    sys.path.insert(0, str(ML_ROOT))

from src.regquest_ml.common.paths import PROCESSED_DATA_DIR
from src.regquest_ml.data.loader import load_processed_data
from src.regquest_ml.features.daily_demand import build_daily_features_table
from src.regquest_ml.forecasting.predict import FEATURE_KEYS

daily_ts = load_processed_data(str(PROCESSED_DATA_DIR / 'daily_time_series.csv'))
daily_features = build_daily_features_table(daily_ts)
X = np.array([[r[k] for k in FEATURE_KEYS] for r in daily_features])
y = np.array([r['target_requests'] for r in daily_features])

print('Random Forest Depth Sensitivity:')
for depth in [2, 3, 4, 6, 8]:
    rf = RandomForestRegressor(n_estimators=50, max_depth=depth, random_state=42)
    rf.fit(X, y)
    mae = np.mean(np.abs(y - rf.predict(X)))
    print(f'  - Max Depth {depth}: In-sample MAE = {mae:.2f} requests/day')


In [ ]:
rf = RandomForestRegressor(n_estimators=50, max_depth=4, random_state=42)
rf.fit(X, y)
importances = sorted(zip(FEATURE_KEYS, rf.feature_importances_), key=lambda x: x[1], reverse=True)
print('\nTop 5 Feature Importances:')
for feat, imp in importances[:5]:
    print(f'  - {feat:<22}: {imp*100:5.1f}%')
